In [1]:
# ------------------------------------------
# Was tut dieses Skript?
# Dieses Skript verarbeitet Mapillary-Coverage-Daten für OSM-Highways in Deutschland.
# Es liest die Coverage-Daten für "pano" und "regular", filtert sie nach einem Coverage-Ratio >= 0.6,
# kombiniert die Ergebnisse, entfernt Duplikate (bevorzugt "pano"), und speichert die finale Tabelle als CSV.
# siehe https://github.com/vizsim/mapillary_coverage/issues/1
# ------------------------------------------

In [2]:
import pandas as pd
import geopandas as gpd

from datetime import datetime
from pathlib import Path
import json
import glob

from config import TILES_CONFIG, PROCESSING_CONFIG, MAPILLARY_CONFIG, GEOFABRIK_CONFIG

In [3]:
# Lese Metadaten aus JSON-Dateien
files = {
    "osm_metadata.json": "osm_data_from",
    "ml_metadata.json": "ml_data_from",
}

# set target variables
osm_data_from = None
ml_data_from = None

for fname, primary_key in files.items():
    p = Path(fname)
    if not p.exists():
        print(f"{p} not found")
        continue

    with p.open("r", encoding="utf-8") as f:
        meta = json.load(f)

    if primary_key in meta:
        value = meta[primary_key]
        if primary_key == "osm_data_from":
            osm_data_from = value
        elif primary_key == "ml_data_from":
            ml_data_from = value
        #print(f"{p.name} -> {primary_key}: {value}")

# optional: show assigned values
print("osm_data_from: ", osm_data_from)
print("ml_data_from: ", ml_data_from)

osm_data_from:  2025-12-04T21:21:15Z
ml_data_from:  2025-12-05T10:58:35.783361+00:00


In [4]:
# Finde alle Bundesland-spezifischen Parquet-Dateien
OUTPUT_FOLDER = PROCESSING_CONFIG["output_folder"]

pano_files = glob.glob(f"{OUTPUT_FOLDER}/DE-*_osm-highways_mp_pano_coverage_latest.parquet")
regular_files = glob.glob(f"{OUTPUT_FOLDER}/DE-*_osm-highways_mp_regular_coverage_latest.parquet")

# Filter Bundesländer basierend auf config.py
selected_bundeslaender = GEOFABRIK_CONFIG.get("bundeslaender")
if selected_bundeslaender:
    # Nur die ausgewählten Bundesländer verarbeiten
    pano_files = [f for f in pano_files if any(bl in f for bl in selected_bundeslaender)]
    regular_files = [f for f in regular_files if any(bl in f for bl in selected_bundeslaender)]
    print(f"🎯 Verarbeite {len(selected_bundeslaender)} ausgewählte Bundesländer: {', '.join(selected_bundeslaender)}")
else:
    print(f"📍 Verarbeite alle gefundenen Bundesländer")

print(f"Gefundene Pano-Dateien: {len(pano_files)}")
print(f"Gefundene Regular-Dateien: {len(regular_files)}")

# Sammle alle Bundesländer
all_pano_dfs = []
all_regular_dfs = []

# Lade alle Pano-Dateien
for file in pano_files:
    df = pd.read_parquet(file)
    all_pano_dfs.append(df)
    print(f"  Pano: {file.split('/')[-1]} → {len(df):,} Zeilen")

# Lade alle Regular-Dateien  
for file in regular_files:
    df = pd.read_parquet(file)
    all_regular_dfs.append(df)
    print(f"  Regular: {file.split('/')[-1]} → {len(df):,} Zeilen")

# Kombiniere alle Bundesländer
if all_pano_dfs:
    gdf_mp_pano = pd.concat(all_pano_dfs, ignore_index=True)
    print(f"\n✓ Gesamt Pano: {len(gdf_mp_pano):,} Zeilen")
else:
    gdf_mp_pano = pd.DataFrame(columns=["osm_id", "mp_coverage_ratio"])
    print("\n⚠️  Keine Pano-Daten gefunden!")

if all_regular_dfs:
    gdf_mp_regular = pd.concat(all_regular_dfs, ignore_index=True)
    print(f"✓ Gesamt Regular: {len(gdf_mp_regular):,} Zeilen")
else:
    gdf_mp_regular = pd.DataFrame(columns=["osm_id", "mp_coverage_ratio"])
    print("⚠️  Keine Regular-Daten gefunden!")

🎯 Verarbeite 1 ausgewählte Bundesländer: DE-HH
Gefundene Pano-Dateien: 1
Gefundene Regular-Dateien: 1
  Pano: DE-HH_osm-highways_mp_pano_coverage_latest.parquet → 3,495 Zeilen
  Regular: DE-HH_osm-highways_mp_regular_coverage_latest.parquet → 80,052 Zeilen

✓ Gesamt Pano: 3,495 Zeilen
✓ Gesamt Regular: 80,052 Zeilen


### pano

In [5]:

# drop all columns except osm_id and mp_coverage_ratio
gdf_mp_pano=gdf_mp_pano[["osm_id","mp_coverage_ratio"]].copy()

# filter for coverage ratio >= threshold
gdf_mp_pano_above_threshold=gdf_mp_pano[gdf_mp_pano["mp_coverage_ratio"]>=PROCESSING_CONFIG['mp_coverage_ratio_threshold']].copy()

# add new column "mapillary_coverage" with value "pano"
gdf_mp_pano_above_threshold["mapillary_coverage"] = "pano"

gdf_mp_pano_above_threshold

,osm_id,mp_coverage_ratio,mapillary_coverage
0,1978,1.000000,pano
7,2483295,0.701443,pano
8,2483296,1.000000,pano
13,2680676,0.896269,pano
14,2680678,0.978139,pano
...,...,...,...
3487,1444409674,1.000000,pano
3488,1444409675,0.981062,pano
3489,1444409676,1.000000,pano
3490,1444779435,0.692601,pano


### regular

In [6]:

# drop all columns except osm_id and mp_coverage_ratio
gdf_mp_regular=gdf_mp_regular[["osm_id","mp_coverage_ratio"]].copy()

# filter for coverage ratio >= threshold
gdf_mp_regular_above_threshold=gdf_mp_regular[gdf_mp_regular["mp_coverage_ratio"]>=PROCESSING_CONFIG['mp_coverage_ratio_threshold']].copy()

# add new column "mapillary_coverage" with value "regular"
gdf_mp_regular_above_threshold["mapillary_coverage"] = "regular"

gdf_mp_regular_above_threshold

,osm_id,mp_coverage_ratio,mapillary_coverage
0,1978,1.000000,regular
2,1880871,1.000000,regular
5,2481205,0.872039,regular
6,2481207,1.000000,regular
8,2481214,0.690862,regular
...,...,...,...
80047,1455895068,1.000000,regular
80048,1455895069,1.000000,regular
80049,1455895070,1.000000,regular
80050,1455895071,1.000000,regular


In [7]:
# Concatenate both GeoDataFrames
both_concat=pd.concat([gdf_mp_pano_above_threshold,gdf_mp_regular_above_threshold],ignore_index=True)

both_concat["mapillary_coverage"].value_counts()

mapillary_coverage
regular    51071
pano        2018
Name: count, dtype: int64

In [8]:
## drop duplicates, keep pano over regular
both_concat = both_concat.sort_values(by="mapillary_coverage", ascending=True).drop_duplicates(subset="osm_id", keep="first")

both_concat["mapillary_coverage"].value_counts()

mapillary_coverage
regular    49434
pano        2018
Name: count, dtype: int64

In [9]:
# check for duplicates
#both_concat["osm_id"].value_counts()

In [10]:
# drop all columns except osm_id and mapillary_coverage
both_concat=both_concat[["osm_id","mapillary_coverage"]].copy()
both_concat

,osm_id,mapillary_coverage
0,1978,pano
1353,914827521,pano
1352,914827518,pano
1351,914827509,pano
1350,914827508,pano
...,...,...
19044,362888813,regular
19045,362906064,regular
19046,362906066,regular
19048,362906070,regular


In [11]:
# Save to CSV
#both_concat.to_csv("output/germany_osm-highways_25-10-06_mp_coverage_23-01-01until25-10-07_ratio_above_06.csv",index=False)
both_concat.to_csv("output/germany_osm-highways_mp-coverage_latest.csv",index=False)




In [12]:
# Get the current date
current_date = datetime.now().strftime("%Y-%m-%d")

# Create the content for the README file
readme_content = f"""# Mapillary Coverage per OSM Highway — Output

This folder contains the **latest** output file for *Mapillary coverage per OSM highway analysis*.

**Data created:** {current_date}<br>
**OSM data:** {osm_data_from.split('T')[0]}<br>
**Mapillary data:** {PROCESSING_CONFIG['min_capture_date']} → {ml_data_from.split('T')[0]}<br>
**Buffer distance:** {PROCESSING_CONFIG['buffer_distance']} meters<br>
**Coverage ratio threshold:** {PROCESSING_CONFIG['mp_coverage_ratio_threshold']} (60%)

Segments are considered *covered* if at least {int(PROCESSING_CONFIG['mp_coverage_ratio_threshold']*100)}% of their length falls within {PROCESSING_CONFIG['buffer_distance']} meters of Mapillary images.
"""



# Write the README file
with open("output/README.md", "w", encoding="utf-8") as readme_file:
    readme_file.write(readme_content)